In [1]:
# Import required libraries
import geopandas as gpd, pandas as pd, numpy as np
from scipy.spatial import Delaunay
from shapely.geometry import Point, Polygon
from pathlib import Path
from collections import Counter
from sklearn.cluster import DBSCAN

In [ ]:
# Generate blankspot from palm geolocation
class BlankspotGenerator():

    def __init__(self, in_count_path, in_boundary_path):
        """ 
        The BlankspotGenerator auomatically generates spots in planting gaps within suitable planting areas.
        This is achived by creating trinagles from the count geolocation filtering out any non-equidistant edges.
        New points are then created to form a parallelogram from the triangles.
        The initial census points are updated to incoperate the new points i.e. grow the points
        The process is iterated until all areas are marked wrt the planting distance.
        Finally, the resulting point gdf is extracted from the initial count to get final blank points.

        Args:
        in_count_path: path to palm count geojson data
        in_boundary_path: path containing aoi boundary data
        """

        self.count_path = Path(in_count_path)
        self.boundary_path = Path(in_boundary_path)

        # create directories for boundary_data and project_files
        parent_path = self.count_path.resolve().parent

        # create blankspot paths
        self.blankspot_path = parent_path / 'Blankspot_files'
        self.blankspot_path.mkdir(parents=True, exist_ok=True)


    # Function to check if a row has a unique edge
    def has_unique_edge(self, row, unique_edges_set):
        a, b, c = row['a'], row['b'], row['c']
        edges = {
            tuple(sorted([a, b])),
            tuple(sorted([b, c])),
            tuple(sorted([c, a]))
        }
        return bool(edges & unique_edges_set)

    # function to rearrangle triangle vertices columns a, b, c based on unique_point_tuples
    def rearrange_row(self, row, unique_point_tuples):
        triangle_points = [row['a'], row['b'], row['c']]
        
        # Check each unique tuple to see if its order can match
        for (a, b) in unique_point_tuples:
            if a in triangle_points and b in triangle_points:
                # Get indices of a and b in the triangle and rearrange accordingly
                idx_a, idx_b = triangle_points.index(a), triangle_points.index(b)
                if idx_b == (idx_a + 1) % 3:  # b follows a in cyclic order
                    return pd.Series([a, b, triangle_points[(idx_b + 1) % 3], row['geometry']], index=['a', 'b', 'c', 'geometry'])
                else:  # a follows b in cyclic order
                    return pd.Series([b, a, triangle_points[(idx_a + 1) % 3], row['geometry']], index=['a', 'b', 'c', 'geometry'])
        # If no rearrangement is needed, return the original order
        return pd.Series([row['a'], row['b'], row['c'], row['geometry']], index=['a', 'b', 'c', 'geometry'])


    # Function to calculate side lengths
    def side_lengths_from_vertices(self, a, b, c):
        return [
            np.linalg.norm(np.array(a) - np.array(b)),
            np.linalg.norm(np.array(b) - np.array(c)),
            np.linalg.norm(np.array(c) - np.array(a))
        ]


    # Function to check if all side differences are less than 1
    def all_side_differences_less_than_one(self, side_lengths):
        return max(side_lengths) - min(side_lengths) < 1


    # Reflect c about base ab to create d
    def reflect_point_across_line(self, a, b, c):
        """
        Reflects point C across the line defined by points A and B to create point D.
        """
        # Convert tuples to Point objects
        a, b, c = Point(a), Point(b), Point(c)
        
        # Midpoint of A and B
        mx, my = (a.x + b.x) / 2, (a.y + b.y) / 2
        
        # Reflect C to get D
        dx = 2 * mx - c.x
        dy = 2 * my - c.y        
        return Point(dx, dy)


    def cluster_and_average_points(self, points_gdf, distance_threshold):
    
        # Extract coordinates as numpy array
        coords = np.array([[point.x, point.y] for point in points_gdf.geometry])

        # Apply DBSCAN clustering with the specified distance threshold
        db = DBSCAN(eps=distance_threshold, min_samples=1, metric='euclidean').fit(coords)
        points_gdf['cluster'] = db.labels_

        # Calculate the average point (centroid) for each cluster
        centroids = points_gdf.groupby('cluster').apply(
            lambda group: Point(group.geometry.x.mean(), group.geometry.y.mean())
        )

        # Convert the resulting Series to a GeoDataFrame
        centroids_gdf = gpd.GeoDataFrame(centroids, columns=['geometry'], crs=points_gdf.crs)
        return centroids_gdf


    # Function that get points within the aoi
    def clip_points_within_boundary(self, points_gdf, boundary_geojson):
    
        # Load the boundary polygon from GeoJSON    
        boundary_gdf = gpd.read_file(boundary_geojson)

        # Ensure both GeoDataFrames are in the same CRS
        if points_gdf.crs != boundary_gdf.crs:
            points_gdf = points_gdf.to_crs(boundary_gdf.crs)

        # Clip points within the boundary polygon
        clipped_points_gdf = gpd.clip(points_gdf, boundary_gdf)
        return clipped_points_gdf
    
    
    # Function to edit final blank gdf
    def blank_edit(self, initial_gdf, copy_gdf, plant_name):

        # Compute the difference: points in final gdf1 but not in initial_gdf1
        new_points_df = copy_gdf[~copy_gdf.geometry.isin(initial_gdf.geometry)]

        # plant_name = count_file.stem.replace('count', 'blankspot')
        new_points_df['Plant_id'] = plant_name + '_' + (new_points_df.index + 1).astype(str)
        new_points_df['lat'] = new_points_df['geometry'].y
        new_points_df['lon'] = new_points_df['geometry'].x

        new_points_df = new_points_df[['Plant_id', 'lat', 'lon', 'geometry']]
        new_points = gpd.GeoDataFrame(new_points_df, geometry=new_points_df['geometry'], crs=initial_gdf.crs)

        new_points = new_points.drop_duplicates(subset='geometry')
        return new_points


    # Function that takes in count geojson and filter triangles
    def point_triangulation(self, initial_gdf, max_side_length = 13):

        # Load the point count GeoJSON file into a ini_gdf_points
        ini_gdf_points = initial_gdf[['geometry']] # keep only geometry

        # Extract x/y coordinates
        points = np.array(list(zip(ini_gdf_points.geometry.x, ini_gdf_points.geometry.y)))

        # Perform Delaunay triangulation
        delaunay_triangles = Delaunay(points)
        triangles, triangle_vertices = [],[]
        
        # Loop through each triangle and filter by side length
        for simplex in delaunay_triangles.simplices:

            # Get the triangle vertices
            triangle_points = points[simplex]
            
            # Calculate side lengths
            side_lengths = [
                np.linalg.norm(triangle_points[0] - triangle_points[1]),
                np.linalg.norm(triangle_points[1] - triangle_points[2]),
                np.linalg.norm(triangle_points[2] - triangle_points[0])
            ]
            
            # If all sides are less than or equal to max_side_length, add to triangles
            if all(length <= max_side_length for length in side_lengths):
                triangles.append(Polygon(triangle_points))
                triangle_vertices.append([tuple(triangle_points[0]), tuple(triangle_points[1]), tuple(triangle_points[2])])

        # Create GeoDataFrame for triangles with vertices info
        gdf_triangles = gpd.GeoDataFrame({
            'geometry': triangles,
            'a': [v[0] for v in triangle_vertices],
            'b': [v[1] for v in triangle_vertices],
            'c': [v[2] for v in triangle_vertices]
        }, crs=ini_gdf_points.crs)
        return gdf_triangles

    
    # Function to get only triangle sides facing blank areas
    def triangle_filter(self, triangulated_gdf):

        # Step 1: Count occurrences of each edge
        edge_counter = Counter()
        for idx, row in triangulated_gdf.iterrows():
            a, b, c = row['a'], row['b'], row['c']
            edges = [
                tuple(sorted([a, b])),
                tuple(sorted([b, c])),
                tuple(sorted([c, a]))
            ]
            edge_counter.update(edges)

        # Step 2: Identify unique edges
        unique_edges = [edge for edge, count in edge_counter.items() if count == 1]
        unique_edges_set = set(unique_edges)

        # Step 3: Filter triangles with unique edges
        gdf_triangles_with_unique_edges = triangulated_gdf[
            triangulated_gdf.apply(lambda row: self.has_unique_edge(row, unique_edges_set), axis=1)]

        # rearrange triangle vertices with as ab just as uniques edge set
        gdf_triangles_with_unique_edges[['a', 'b', 'c', 'geometry']] = gdf_triangles_with_unique_edges.apply(
            self.rearrange_row, unique_point_tuples=unique_edges_set, axis=1)

        # Step 4: Calculate side lengths for each triangle with unique edges and filter
        gdf_triangles_with_unique_edges['side_lengths'] = gdf_triangles_with_unique_edges.apply(
            lambda row: self.side_lengths_from_vertices(row['a'], row['b'], row['c']), axis=1)

        # Step 5: Final filter based on side length differences
        filtered_gdf = gdf_triangles_with_unique_edges[
            gdf_triangles_with_unique_edges['side_lengths'].apply(self.all_side_differences_less_than_one)
        ]
        
        # Drop auxiliary column and return final filtered GeoDataFrame
        filtered_gdf = filtered_gdf.drop(columns=['side_lengths'])        
        return filtered_gdf

    # Function to generate blank points
    def blank_points(self, in_filtered_gdf, boundary_dir):

        # Calculate D points and create a new GeoDataFrame with them
        in_filtered_gdf['d'] = in_filtered_gdf.apply(lambda row: self.reflect_point_across_line(row['a'], row['b'], row['c']), axis=1)
        d_points_gdf = gpd.GeoDataFrame(in_filtered_gdf[['d']], geometry='d', crs=in_filtered_gdf.crs)

        # cluster and average close points
        centroids_gdf = self.cluster_and_average_points(d_points_gdf, distance_threshold=6)

        # clip-off any point outside field boundary
        blankspot_aoi = self.clip_points_within_boundary(centroids_gdf, boundary_dir)
        return blankspot_aoi
    
    
    def execute_class(self):
        # iterate palm count dir for palm geojsons
        for count_file in self.count_path.iterdir():
            print(count_file)

            ini_gdf_points = gpd.read_file(count_file)

            # Create a copy for updating during the loop
            gdf1 = ini_gdf_points.copy()

            while True:                            
                # create triangles from geolocated points
                in_tri = self.point_triangulation(gdf1)

                # filter formed triangles
                filtered_tri = self.triangle_filter(in_tri)

                # generate spots
                boundary_path = self.boundary_path / (count_file.stem.replace('count', 'boundary') + '.geojson')
                blankspot = self.blank_points(filtered_tri, boundary_path)

                # get length
                print(len(blankspot))

                # Break the loop if there are no more triangles after filtering
                if blankspot.empty:

                    plant_name = count_file.stem.replace('count', 'blankspot')

                    new_points = self.blank_edit(ini_gdf_points, gdf1, plant_name)

                    # save blankspot
                    dest_path = f'{self.blankspot_path}/{plant_name}.geojson'
                    new_points.to_file(dest_path, driver='GeoJSON')
                    break
    
                # Update gdf1 by adding blankspot points
                gdf1 = gpd.GeoDataFrame(pd.concat([gdf1, blankspot], ignore_index=True), crs=ini_gdf_points.crs)


# implementation
count_path = '.../Project_folder/GeoTIFF_images/'
boundary_path = '.../Project_folder/polygon_data/'

Blankspot_Generator = BlankspotGenerator(count_path, boundary_path)
Blankspot_Generator.execute_class()
